# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa — Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All references to dataset structure use the Croissant `@id` fields for clarity and reproducibility.

### Dataset Source

The dataset's croissant schema is available at this URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

We'll load the dataset metadata and show a description using the `mlcroissant` library. This provides access to all record sets, fields, and additional metadata defined in the Croissant file.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Retrieve and pretty-print the metadata
meta = dataset.metadata
print(f"Dataset Name: {meta.name}")
print(f"Description: {meta.description}")
print(f"Version: {meta.version}")
print(f"Identifier: {meta.identifier}")

## 2. Data Overview

Let's get an overview of the main record sets, their fields, and the associated IDs as defined in the schema. We'll reference all entities by their Croissant `@id`.

**Record sets and fields are the main entry points to working with data in mlcroissant.**


In [ ]:
# List available record sets and fields by @id

print("Available Record Sets:")
for rs in dataset.record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs['name']}")

# For demonstration, print their fields and columns as well
print("\nRecord Set Fields and Columns:")
for rs in dataset.record_sets:
    print(f"Record Set: {rs['name']} (@id: {rs['@id']})")
    if 'field' in rs:
        for f in rs['field']:
            if isinstance(f, dict):
                fid = f.get('@id', str(f))
                fname = f.get('name', '')
            else:
                fid = f
                fname = ''
            print(f"    - Field @id: {fid} {f'({fname})' if fname else ''}")
    if 'column' in rs:
        for c in rs['column']:
            if isinstance(c, dict):
                cid = c.get('@id', str(c))
                cname = c.get('name', '')
            else:
                cid = c
                cname = ''
            print(f"    - Column @id: {cid} {f'({cname})' if cname else ''}")

## 3. Data Extraction

Now, let's load data from a record set. 

First, we'll gather the list of available record set `@id`s. **You can replace these variables to focus on others later.**

We use the `records()` function with the `record_set` parameter set to the `@id` of your chosen record set.

In [ ]:
# Get record set @id list
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Record Set IDs:", record_set_ids)

# Choose the primary data table (record set) for the main analysis
# You can change the following to any available record set
main_record_set_id = record_set_ids[0]  # e.g. 'cr:RecordSet' if present

# Load records for the selected record set
records = list(dataset.records(record_set=main_record_set_id))
# Convert to pandas DataFrame for analysis
df = pd.DataFrame(records)

print(f"Loaded {len(df)} records from record set {main_record_set_id}")
print("Columns:", list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's explore and preprocess the data using common EDA steps such as filtering, normalizing, and grouping. All column references use their Croissant `@id` fields.

**Tip:** Use the field list from above to find an appropriate numeric field (e.g., scores from PHQ-9, GAD-7, or PSQ).


In [ ]:
# Select a numeric field for EDA by field @id (edit as needed)
# Example guesses: 'gad7_score', 'phq9_score', 'psq_score' (these must be the @id or column names in your dataset)
# For demo, try to automatically pick a field containing 'score'.
numeric_field = None
for col in df.columns:
    if 'score' in col.lower():
        numeric_field = col
        break
if numeric_field is None:
    # fallback to first numeric field
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
print(f"Selected numeric field for analysis: {numeric_field}")

# Set threshold for filtering (change as appropriate)
threshold = 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}: {filtered_df.shape[0]} rows")
filtered_df.head()

In [ ]:
# Normalize the chosen numeric field
norm_field = f"{numeric_field}_normalized"
filtered_df[norm_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized values for {numeric_field} (first few rows):")
filtered_df[[numeric_field, norm_field]].head()

In [ ]:
# Optionally group by a categorical field (e.g., 'gender', 'village', etc.), using field @id or column name
# We'll try to auto-select a suitable group field
group_field = None
for candidate in ['gender', 'village', 'level_of_education', 'marital_status']:
    for col in df.columns:
        if candidate in col.lower():
            group_field = col
            break
    if group_field:
        break

if group_field:
    print(f"Grouping by field: {group_field}\n")
    grouped = filtered_df.groupby(group_field)[numeric_field].mean()
    print(grouped.head())
else:
    print("No suitable grouping field found.")

## 5. Visualization

Let's visualize the distribution of the chosen numeric field, and if available, compare distributions by a categorical group.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the primary numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
plt.title(f"{numeric_field} Distribution")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# Boxplot by group if possible
if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- This notebook demonstrated the use of [`mlcroissant`](https://pypi.org/project/mlcroissant/) for structured, reproducible exploration of the FAIR² Kilifi mental health survey dataset using only Croissant `@id` references.
- We loaded metadata and records, reviewed the dataset's table structure, and analyzed mental health survey scores.
- Further steps could include advanced feature engineering, machine learning, or combining with other health/social datasets.
